In [ ]:
# Kaggle setup
# The YOLOv7 requirements file is old. If one optional package fails to build,
# we keep going because Kaggle already has the main runtime packages installed.
!pip install -q wandb pyyaml seaborn opencv-python

from pathlib import Path
import subprocess

YOLOV7_DIR_SETUP = Path("/kaggle/working/yolov7")

if not YOLOV7_DIR_SETUP.exists():
    subprocess.run(
        ["git", "clone", "--quiet", "https://github.com/WongKinYiu/yolov7.git", str(YOLOV7_DIR_SETUP)],
        check=True,
    )
else:
    print("YOLOv7 repo already exists:", YOLOV7_DIR_SETUP)

req_result = subprocess.run(
    ["pip", "install", "-q", "-r", str(YOLOV7_DIR_SETUP / "requirements.txt")]
)

if req_result.returncode != 0:
    print("YOLOv7 requirements install had a non-critical package issue; continuing with Kaggle defaults.")

# Kaggle's newest CUDA/PyTorch image no longer contains kernels for the P100
# (compute capability 6.0), and PyTorch 2.6+ also changed torch.load's default.
# This CUDA 12.1 build supports the P100 and keeps YOLOv7 checkpoint loading compatible.
subprocess.run(
    [
        "pip", "install", "-q", "--upgrade", "--force-reinstall",
        "torch==2.5.1", "torchvision==0.20.1",
        "--index-url", "https://download.pytorch.org/whl/cu121",
    ],
    check=True,
)

# Be explicit at every YOLOv7 checkpoint load site. These official YOLOv7
# checkpoints contain a serialized Model object and therefore require
# weights_only=False. Only use checkpoints from a source you trust.
for script_name in ("train.py", "test.py", "detect.py"):
    script_path = YOLOV7_DIR_SETUP / script_name
    if not script_path.exists():
        continue
    source = script_path.read_text()
    source = source.replace("torch.load(weights, map_location=device)",
                            "torch.load(weights, map_location=device, weights_only=False)")
    source = source.replace("torch.load(w, map_location=device)",
                            "torch.load(w, map_location=device, weights_only=False)")
    script_path.write_text(source)

# NumPy 2.4 removed the deprecated np.trapz name used by YOLOv7 when
# computing average precision. np.trapezoid is its direct replacement.
metrics_path = YOLOV7_DIR_SETUP / "utils" / "metrics.py"
metrics_source = metrics_path.read_text()
metrics_source = metrics_source.replace("np.trapz(", "np.trapezoid(")
metrics_path.write_text(metrics_source)


In [ ]:
import os
import sys
import random
import shutil
import subprocess
import re
from pathlib import Path
from collections import defaultdict

import yaml
import wandb
import torch
import numpy as np

# Compatibility for older YOLOv7 code on newer Kaggle NumPy versions.
for alias, value in {"int": int, "float": float, "bool": bool}.items():
    if not hasattr(np, alias):
        setattr(np, alias, value)

YOLOV7_DIR = Path("/kaggle/working/yolov7")
sys.path.insert(0, str(YOLOV7_DIR))


In [ ]:
# Run identity
MODEL_WEIGHTS = "yolov7.pt"
MODEL_WEIGHTS_URL = "https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt"
MODEL_CFG = "yolov7.yaml"
RUN_NAME = "RunI_yolov7_imgsz640"
WANDB_PROJECT = "smallDefectDetection"

# Dataset structure
DATASET_NAMES = [
    "DAGM",
    "GC10-DET",
    "KolektorSDD2",
    "MPDD",
    "MTD",
    "Severstal",
    "VisA",
]

SIZE_BUCKETS = ["small", "medium", "large"]

# Split settings
SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# YOLO training hyperparameters
IMG_SIZE = 640
EPOCHS = 100
BATCH_SIZE = 8
PATIENCE = 25
WORKERS = 2
DEVICE = 0

# Paths on Kaggle
BASE_DIR = Path("/kaggle/working")
PREPARED_DATASET_DIR = BASE_DIR / "run_a_yolo_dataset"


In [ ]:
def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        return user_secrets.get_secret(name)
    except Exception:
        return os.environ.get(name)


WANDB_API_KEY = get_secret("wandb_api_key")

if WANDB_API_KEY is None:
    raise ValueError("WANDB_API_KEY not found. Add it in Kaggle Secrets.")

wandb.login(key=WANDB_API_KEY)


In [ ]:
KAGGLE_INPUT_ROOT = Path("/kaggle/input")

expected_datasets = set(DATASET_NAMES)

print("Searching for dataset root under:", KAGGLE_INPUT_ROOT)

valid_roots = []

for root, dirs, files in os.walk(KAGGLE_INPUT_ROOT):
    root_path = Path(root)
    dir_set = set(dirs)

    matches = expected_datasets.intersection(dir_set)

    if len(matches) >= 5:
        valid_roots.append(root_path)
        print("Found candidate:", root_path)
        print("Matches:", sorted(matches))

if not valid_roots:
    raise FileNotFoundError(
        "Could not find a folder containing the expected dataset folders: "
        f"{sorted(expected_datasets)}"
    )

hf_dataset_path = valid_roots[0]

print("\nUsing dataset root:", hf_dataset_path)
print("Available datasets:", sorted([p.name for p in hf_dataset_path.iterdir() if p.is_dir()]))


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

samples = []
missing_count = 0

for dataset_name in DATASET_NAMES:
    for size_bucket in SIZE_BUCKETS:
        image_dir = hf_dataset_path / dataset_name / size_bucket / "images"
        label_dir = hf_dataset_path / dataset_name / size_bucket / "labels_yolo"

        if not image_dir.exists() or not label_dir.exists():
            print("Missing directory:", dataset_name, size_bucket)
            continue

        label_index = {
            label_path.stem: label_path
            for label_path in label_dir.glob("*.txt")
        }

        matched = 0
        bucket_missing = 0

        for image_path in image_dir.iterdir():
            if image_path.suffix.lower() not in IMAGE_EXTS:
                continue

            image_stem = image_path.stem
            base_stem = image_stem.removesuffix("_defect")

            possible_label_stems = [
                image_stem,
                image_stem.replace("_defect", "_bbs"),
                f"{base_stem}_bbs",
            ]

            label_path = next(
                (
                    label_index[stem]
                    for stem in possible_label_stems
                    if stem in label_index
                ),
                None,
            )

            if label_path is None:
                bucket_missing += 1
                continue

            samples.append({
                "image_path": image_path,
                "label_path": label_path,
                "dataset": dataset_name,
                "size": size_bucket,
                "stratum": f"{dataset_name}_{size_bucket}",
            })

            matched += 1

        missing_count += bucket_missing
        print(
            f"{dataset_name}/{size_bucket}: "
            f"{matched} matched, {bucket_missing} missing"
        )

print("\nTotal usable samples:", len(samples))
print("Total missing labels:", missing_count)

if not samples:
    raise RuntimeError("No image-label pairs were matched.")


In [ ]:
import random
from collections import defaultdict

random.seed(SEED)

by_stratum = defaultdict(list)

for sample in samples:
    by_stratum[sample["stratum"]].append(sample)

train_samples = []
val_samples = []
test_samples = []

for stratum, group in sorted(by_stratum.items()):
    random.shuffle(group)

    n = len(group)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    train_samples.extend(group[:n_train])
    val_samples.extend(group[n_train:n_train + n_val])
    test_samples.extend(group[n_train + n_val:])

random.shuffle(train_samples)
random.shuffle(val_samples)
random.shuffle(test_samples)

print("Train:", len(train_samples))
print("Val:", len(val_samples))
print("Test:", len(test_samples))
print("Total:", len(train_samples) + len(val_samples) + len(test_samples))

assert len(train_samples) > 0
assert len(val_samples) > 0
assert len(test_samples) > 0


In [ ]:
from collections import Counter

def count_by_size(samples, split_name):
    counts = Counter(s["size"] for s in samples)
    total = len(samples)

    print()
    print(split_name)
    print("Total :", total)
    print("Small :", counts["small"])
    print("Medium:", counts["medium"])
    print("Large :", counts["large"])

count_by_size(train_samples, "Train")
count_by_size(val_samples, "Val")
count_by_size(test_samples, "Test")


In [ ]:
def reset_dir(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


reset_dir(PREPARED_DATASET_DIR)

splits_to_make = [
    "train",
    "val",
    "test",
    "test_small",
    "test_medium",
    "test_large",
]

for split in splits_to_make:
    (PREPARED_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (PREPARED_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Prepared dataset dir:", PREPARED_DATASET_DIR)


In [ ]:
def rewrite_label_as_single_class(src_label_path, dst_label_path):
    new_lines = []

    with open(src_label_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) < 5:
                continue

            coords = parts[1:5]
            new_lines.append("0 " + " ".join(coords))

    with open(dst_label_path, "w") as f:
        f.write("\n".join(new_lines))


def export_split(split_name, split_samples):
    for idx, sample in enumerate(split_samples):
        src_image = sample["image_path"]
        src_label = sample["label_path"]

        safe_name = f"{sample['dataset']}_{sample['size']}_{idx}_{src_image.name}"

        dst_image = PREPARED_DATASET_DIR / "images" / split_name / safe_name
        dst_label = PREPARED_DATASET_DIR / "labels" / split_name / f"{Path(safe_name).stem}.txt"

        shutil.copy2(src_image, dst_image)
        rewrite_label_as_single_class(src_label, dst_label)


export_split("train", train_samples)
export_split("val", val_samples)
export_split("test", test_samples)

test_small_samples = [s for s in test_samples if s["size"] == "small"]
test_medium_samples = [s for s in test_samples if s["size"] == "medium"]
test_large_samples = [s for s in test_samples if s["size"] == "large"]

export_split("test_small", test_small_samples)
export_split("test_medium", test_medium_samples)
export_split("test_large", test_large_samples)

print("YOLO dataset prepared.")
print("Test overall:", len(test_samples))
print("Test small:", len(test_small_samples))
print("Test medium:", len(test_medium_samples))
print("Test large:", len(test_large_samples))


In [ ]:
def write_data_yaml(path, test_split):
    # YOLOv7 predates the modern `path` YAML key and resolves relative split
    # paths from the repository working directory. Use absolute paths here.
    data_yaml = {
        "train": str(PREPARED_DATASET_DIR / "images" / "train"),
        "val": str(PREPARED_DATASET_DIR / "images" / "val"),
        "test": str(PREPARED_DATASET_DIR / "images" / test_split),
        "nc": 1,
        "names": ["defect"],
    }

    with open(path, "w") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

data_yaml_path = PREPARED_DATASET_DIR / "data.yaml"

data_yaml_all = Path("/kaggle/working/data_all.yaml")
data_yaml_small = Path("/kaggle/working/data_small.yaml")
data_yaml_medium = Path("/kaggle/working/data_medium.yaml")
data_yaml_large = Path("/kaggle/working/data_large.yaml")

write_data_yaml(data_yaml_path, "test")
write_data_yaml(data_yaml_all, "test")
write_data_yaml(data_yaml_small, "test_small")
write_data_yaml(data_yaml_medium, "test_medium")
write_data_yaml(data_yaml_large, "test_large")

print(data_yaml_path)
print(data_yaml_path.read_text())


In [ ]:
for split in ["train", "val", "test", "test_small", "test_medium", "test_large"]:
    image_count = len(list((PREPARED_DATASET_DIR / "images" / split).glob("*")))
    label_count = len(list((PREPARED_DATASET_DIR / "labels" / split).glob("*.txt")))

    print(split)
    print(" images:", image_count)
    print(" labels:", label_count)


In [ ]:
WEIGHTS_PATH = YOLOV7_DIR / MODEL_WEIGHTS

if not WEIGHTS_PATH.exists():
    subprocess.run(
        ["wget", "-q", MODEL_WEIGHTS_URL, "-O", str(WEIGHTS_PATH)],
        check=True,
    )

print("Weights ready at:", WEIGHTS_PATH, WEIGHTS_PATH.exists())


In [ ]:
train_cmd = [
    "python", str(YOLOV7_DIR / "train.py"),
    "--weights", str(WEIGHTS_PATH),
    "--cfg", str(YOLOV7_DIR / "cfg" / "training" / MODEL_CFG),
    "--data", str(data_yaml_path),
    "--hyp", str(YOLOV7_DIR / "data" / "hyp.scratch.tiny.yaml"
                 if "tiny" in MODEL_CFG else YOLOV7_DIR / "data" / "hyp.scratch.p5.yaml"),
    "--img-size", str(IMG_SIZE), str(IMG_SIZE),
    "--batch-size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--workers", str(WORKERS),
    "--device", str(DEVICE),
    "--project", "runs/train",
    "--name", RUN_NAME,
    "--exist-ok",
]

print("Running:", " ".join(train_cmd))

result = subprocess.run(train_cmd, cwd=str(YOLOV7_DIR))
if result.returncode != 0:
    raise RuntimeError(f"YOLOv7 training failed with exit code {result.returncode}")


In [ ]:
best_model_path = Path("/kaggle/working/yolov7/runs/train") / RUN_NAME / "weights" / "best.pt"

if not best_model_path.exists():
    candidates = sorted(Path("/kaggle/working").glob(f"**/{RUN_NAME}/weights/best.pt"))
    print("Found best.pt candidates:")
    for p in candidates:
        print(p)

    if not candidates:
        raise FileNotFoundError("No best.pt found. Training did not finish/save correctly.")

    best_model_path = candidates[0]

print("Using:", best_model_path)


In [ ]:
# Import the repo's own test.py as a module so we can call test.test()
# directly and get metrics back as Python values (same idea as calling
# best_model.val() in the ultralytics API).
os.chdir(str(YOLOV7_DIR))
import test as yolov7_test
import importlib
importlib.reload(yolov7_test)


In [ ]:
val_results = yolov7_test.test(
    data=str(data_yaml_all),
    weights=str(best_model_path),
    batch_size=BATCH_SIZE,
    imgsz=IMG_SIZE,
    conf_thres=0.001,
    iou_thres=0.65,
    task="val",
    save_json=False,
    plots=True,
    project="runs/test",
    name=f"{RUN_NAME}_val",
    exist_ok=True,
)

print(val_results)


In [ ]:
test_sets = {
    "overall": data_yaml_all,
    "small": data_yaml_small,
    "medium": data_yaml_medium,
    "large": data_yaml_large,
}

test_results = {}

for test_name, yaml_path in test_sets.items():
    print()
    print("Running test evaluation:", test_name)

    results, maps, times = yolov7_test.test(
        data=str(yaml_path),
        weights=str(best_model_path),
        batch_size=BATCH_SIZE,
        imgsz=IMG_SIZE,
        conf_thres=0.001,
        iou_thres=0.65,
        task="test",
        save_json=False,
        plots=True,
        project="runs/test",
        name=f"{RUN_NAME}_test_{test_name}",
        exist_ok=True,
    )

    # results tuple layout from yolov7 test.py: (mp, mr, map50, map, ...)
    mp, mr, map50, map_all = results[0], results[1], results[2], results[3]

    test_results[test_name] = {
        "Precision": mp,
        "Recall": mr,
        "mAP50": map50,
        "mAP50_95": map_all,
        "speed_ms": times[1] * 1000 if times is not None else float("nan"),
    }

    print(test_results[test_name])


In [ ]:
from models.experimental import attempt_load
from utils.datasets import letterbox
from utils.general import non_max_suppression, scale_coords
from utils.torch_utils import select_device
import cv2


def box_iou_xyxy(box1, box2):
    x1 = torch.max(box1[:, 0].unsqueeze(1), box2[:, 0].unsqueeze(0))
    y1 = torch.max(box1[:, 1].unsqueeze(1), box2[:, 1].unsqueeze(0))
    x2 = torch.min(box1[:, 2].unsqueeze(1), box2[:, 2].unsqueeze(0))
    y2 = torch.min(box1[:, 3].unsqueeze(1), box2[:, 3].unsqueeze(0))

    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])
    union = area1.unsqueeze(1) + area2.unsqueeze(0) - inter

    return inter / union.clamp(min=1e-6)


def yolo_to_xyxy(label_path, img_w, img_h):
    boxes = []

    if not Path(label_path).exists():
        return torch.zeros((0, 4))

    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) >= 5:
                cx = float(parts[1])
                cy = float(parts[2])
                w = float(parts[3])
                h = float(parts[4])

                x1 = (cx - w / 2) * img_w
                y1 = (cy - h / 2) * img_h
                x2 = (cx + w / 2) * img_w
                y2 = (cy + h / 2) * img_h

                boxes.append([x1, y1, x2, y2])

    if not boxes:
        return torch.zeros((0, 4))

    return torch.tensor(boxes, dtype=torch.float32)


device = select_device(str(DEVICE))
fp_model = attempt_load(str(best_model_path), map_location=device)
fp_model.eval()
stride = int(fp_model.stride.max())


def run_inference(img_path, imgsz=640, conf_thres=0.25, iou_thres=0.5):
    img0 = cv2.imread(str(img_path))
    img_h0, img_w0 = img0.shape[:2]

    img = letterbox(img0, imgsz, stride=stride)[0]
    img = img[:, :, ::-1].transpose(2, 0, 1)
    img = np.ascontiguousarray(img)
    img_t = torch.from_numpy(img).to(device).float() / 255.0
    if img_t.ndimension() == 3:
        img_t = img_t.unsqueeze(0)

    with torch.no_grad():
        pred = fp_model(img_t)[0]
    pred = non_max_suppression(pred, conf_thres, iou_thres)[0]

    if pred is None or len(pred) == 0:
        return torch.zeros((0, 4)), np.array([]), img_w0, img_h0

    pred[:, :4] = scale_coords(img_t.shape[2:], pred[:, :4], img0.shape).round()
    boxes = pred[:, :4].cpu()
    confs = pred[:, 4].cpu().numpy()

    return boxes, confs, img_w0, img_h0


def compute_fp_per_image(img_dir, lbl_dir, conf=0.25, iou_thresh=0.5, imgsz=640):
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)

    total_fp = 0
    total_images = 0

    image_paths = sorted([
        p for p in img_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    ])

    for img_path in image_paths:
        pred_boxes, conf_scores, img_w, img_h = run_inference(
            img_path, imgsz=imgsz, conf_thres=conf, iou_thres=0.7
        )

        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        gt_boxes = yolo_to_xyxy(lbl_path, img_w, img_h)

        n_pred = len(pred_boxes)
        n_gt = len(gt_boxes)

        if n_pred == 0:
            total_images += 1
            continue

        if n_gt == 0:
            total_fp += n_pred
            total_images += 1
            continue

        iou_matrix = box_iou_xyxy(pred_boxes, gt_boxes)

        matched_gt = set()
        fp_count = 0
        sorted_idx = np.argsort(-conf_scores)

        for pred_idx in sorted_idx:
            iou_with_gts = iou_matrix[pred_idx]
            best_gt_idx = iou_with_gts.argmax().item()
            best_iou = iou_with_gts[best_gt_idx].item()

            if best_iou >= iou_thresh and best_gt_idx not in matched_gt:
                matched_gt.add(best_gt_idx)
            else:
                fp_count += 1

        total_fp += fp_count
        total_images += 1

    fp_per_image = total_fp / total_images if total_images > 0 else 0.0

    return fp_per_image, total_fp, total_images


fp_splits = {
    "overall": "test",
    "small": "test_small",
    "medium": "test_medium",
    "large": "test_large",
}

fp_results = {}

for split_name, folder_name in fp_splits.items():
    fp_per_img, total_fp, total_imgs = compute_fp_per_image(
        img_dir=PREPARED_DATASET_DIR / "images" / folder_name,
        lbl_dir=PREPARED_DATASET_DIR / "labels" / folder_name,
        conf=0.25,
        iou_thresh=0.5,
        imgsz=IMG_SIZE,
    )

    fp_results[split_name] = {
        "fp_per_image": fp_per_img,
        "total_fp": total_fp,
        "total_images": total_imgs,
    }

    print(f"{split_name}:")
    print(f"  Total images : {total_imgs}")
    print(f"  Total FPs    : {total_fp}")
    print(f"  FP per image : {fp_per_img:.4f}")

FP_Per_Image = fp_results["overall"]["fp_per_image"]
print("Overall FP_Per_Image:", FP_Per_Image)


In [ ]:
import pandas as pd

overall = test_results["overall"]
small = test_results["small"]
medium = test_results["medium"]
large = test_results["large"]

inference_time_ms = overall["speed_ms"]

summary_row = {
    "Experiment": RUN_NAME,
    "Model": "YOLOv7",
    "Batch": BATCH_SIZE,
    "Epochs": EPOCHS,
    "mAP50": overall["mAP50"],
    "mAP50_95": overall["mAP50_95"],
    "Precision": overall["Precision"],
    "Recall": overall["Recall"],
    "mAP50_Small": small["mAP50"],
    "mAP50_Medium": medium["mAP50"],
    "mAP50_Large": large["mAP50"],
    "Recall_Small": small["Recall"],
    "Recall_Medium": medium["Recall"],
    "Recall_Large": large["Recall"],
    "Inference_Time_ms": inference_time_ms,
    "FP_Per_Image": FP_Per_Image,
    "Notes": "YOLOv7 (standard) baseline, imgsz 640, official WongKinYiu repo",
}

summary_df = pd.DataFrame([summary_row])
summary_df


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/final_outputs") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(best_model_path, OUTPUT_DIR / "best.pt")
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False)

print("Saved weights to:", OUTPUT_DIR / "best.pt")
print("Saved summary to:", OUTPUT_DIR / "summary.csv")
